In [2]:

# Section 1: Load Original RouterBench Training Data

from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
import joblib

PROJECT_ROOT = Path.home() / "Desktop" / "Erdos-Router"

DATA = (
    PROJECT_ROOT
    / "1_data_preparation"
    / "data"
    / "processed"
    / "char_tfidf_50k"
)

# Load training features
X_train = sparse.load_npz(DATA / "X_train.npz").tocsr()

# Load training costs
cost_train = pd.read_pickle(DATA / "train_costs.pkl")

# Load training questions
train_questions = pd.read_pickle(
    DATA / "train_questions.pkl"
)

# Load preprocessing
preprocessing = joblib.load(
    DATA / "preprocessing.joblib"
)

MODELS = list(cost_train.columns)

print("Training features:", X_train.shape)
print("Training costs:", cost_train.shape)
print("Training questions:", train_questions.shape)
print("Number of LLMs:", len(MODELS))


Training features: (21900, 50001)
Training costs: (21900, 11)
Training questions: (21900, 6)
Number of LLMs: 11


In [3]:

# Section 2.1: Inspect Question Groups

groups = train_questions["question_group"]

print("Number of training questions:", len(train_questions))
print("Number of unique groups:", groups.nunique())

print("\nLargest groups:")
print(groups.value_counts().head(10))

print("\nSmallest group size:", groups.value_counts().min())
print("Largest group size:", groups.value_counts().max())


Number of training questions: 21900
Number of unique groups: 21888

Largest groups:
question_group
26020    2
26228    2
2071     2
2232     2
20881    2
2165     2
261      2
3043     2
3076     2
34153    2
Name: count, dtype: int64

Smallest group size: 1
Largest group size: 2


In [ ]:
# divide the training data set into two smaller data sets (80 : 20)

# Section 2.2: Group-Aware Sub-Training / Sub-Validation Split

import numpy as np
from sklearn.model_selection import GroupShuffleSplit

SEED = 42

groups = train_questions["question_group"].to_numpy()

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

subtrain_idx, subval_idx = next(
    splitter.split(X_train, groups=groups)
)

# Features
X_subtrain = X_train[subtrain_idx]
X_subval = X_train[subval_idx]

# Costs: positional indexing preserves feature-cost alignment
cost_subtrain = cost_train.iloc[subtrain_idx].copy()
cost_subval = cost_train.iloc[subval_idx].copy()

# Verify no group overlap
subtrain_groups = set(groups[subtrain_idx])
subval_groups = set(groups[subval_idx])

assert subtrain_groups.isdisjoint(subval_groups)

# Verify observations are partitioned exactly once
assert len(subtrain_idx) + len(subval_idx) == X_train.shape[0]
assert len(set(subtrain_idx) & set(subval_idx)) == 0

print("Sub-training questions:", X_subtrain.shape[0])
print("Sub-validation questions:", X_subval.shape[0])

print("Sub-training features:", X_subtrain.shape)
print("Sub-validation features:", X_subval.shape)

print("\nGroup overlap:", len(
    subtrain_groups.intersection(subval_groups)
))

print("\nGroup-aware split verified.")


Sub-training questions: 17520
Sub-validation questions: 4380
Sub-training features: (17520, 50001)
Sub-validation features: (4380, 50001)

Group overlap: 0

Group-aware split verified.


In [7]:

# Section 2.3: Save Group-Aware Training Split

from pathlib import Path
import numpy as np

SPLIT_DIR = PROJECT_ROOT / "saved_models" / "training_splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

split_path = SPLIT_DIR / "routerbench_grouped_subtrain_subval_seed42.npz"

# Save the positions within the original training partition
np.savez(
    split_path,
    subtrain_idx=subtrain_idx,
    subval_idx=subval_idx
)

# Verify the saved indices can be loaded
saved_split = np.load(split_path)

np.testing.assert_array_equal(
    saved_split["subtrain_idx"], subtrain_idx
)

np.testing.assert_array_equal(
    saved_split["subval_idx"], subval_idx
)

print("Training split saved successfully.")
print("Saved to:", split_path)


Training split saved successfully.
Saved to: /Users/garenchiloyan/Desktop/Erdos-Router/saved_models/training_splits/routerbench_grouped_subtrain_subval_seed42.npz


In [ ]:
# define loss function using Gamma deviance

# a perfect prediction equals the actual observed value while an imperfect one does not

# Section 3.1: Gamma Deviance Loss

import numpy as np

def gamma_deviance(y_true, y_pred):
    """
    Compute mean Gamma deviance.

    Parameters
    ----------
    y_true : array-like
        Strictly positive observed costs.
    y_pred : array-like
        Strictly positive predicted mean costs.

    Returns
    -------
    float
        Mean Gamma deviance.
    """
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    if y_true.shape != y_pred.shape:
        raise ValueError("Shapes must match.")

    if not np.isfinite(y_true).all():
        raise ValueError("Observed costs must be finite.")

    if not np.isfinite(y_pred).all():
        raise ValueError("Predictions must be finite.")

    if (y_true <= 0).any() or (y_pred <= 0).any():
        raise ValueError("Gamma costs must be strictly positive.")

    ratio = y_true / y_pred

    return float(
        2.0 * np.mean(ratio - 1.0 - np.log(ratio))
    )


# Verify the loss function

actual = np.array([1.0, 2.0, 3.0])

perfect = np.array([1.0, 2.0, 3.0])
imperfect = np.array([1.2, 1.8, 2.5])

print("Perfect predictions:", gamma_deviance(actual, perfect))
print("Imperfect predictions:", gamma_deviance(actual, imperfect))


Perfect predictions: 0.0
Imperfect predictions: 0.02605595252441209


In [9]:

# Section 3.2: Regularized Gamma Regression Objective

import numpy as np

def gamma_objective(y_true, mu_pred, coefficients, lam):
    """
    Compute the regularized Gamma objective:

        0.5 * mean Gamma deviance
        + 0.5 * lam * ||coefficients||^2

    The intercept is excluded from coefficients.
    """
    if lam < 0:
        raise ValueError("Regularization strength must be nonnegative.")

    coefficients = np.asarray(coefficients, dtype=np.float64)

    # Data-fitting term
    data_loss = 0.5 * gamma_deviance(
        y_true,
        mu_pred
    )

    # L2 regularization penalty
    penalty = 0.5 * lam * np.sum(coefficients ** 2)

    return float(data_loss + penalty)


# Verify the implementation on a simple example

actual = np.array([1.0, 2.0, 3.0])
predicted = np.array([1.2, 1.8, 2.5])

beta = np.array([0.5, -0.3, 0.2])
lam = 0.1

print("Gamma deviance:", gamma_deviance(actual, predicted))
print("Half Gamma deviance:", 0.5 * gamma_deviance(actual, predicted))
print("L2 penalty:", 0.5 * lam * np.sum(beta ** 2))
print("Regularized objective:",
      gamma_objective(actual, predicted, beta, lam))


Gamma deviance: 0.02605595252441209
Half Gamma deviance: 0.013027976262206045
L2 penalty: 0.019000000000000003
Regularized objective: 0.03202797626220605


In [10]:

# Section 3.3: Regularized Gamma Objective and Gradient

import numpy as np

def gamma_loss_and_gradient(params, X, y, lam):
    """
    Compute the regularized Gamma objective and gradient.

    params[0]  : intercept
    params[1:] : regression coefficients

    X : sparse CSR matrix or NumPy array
    y : positive observed costs
    lam : nonnegative L2 regularization strength

    Returns
    -------
    loss : float
    gradient : ndarray
    """

    if lam < 0:
        raise ValueError("lam must be nonnegative.")

    y = np.asarray(y, dtype=np.float64)
    params = np.asarray(params, dtype=np.float64)

    n = X.shape[0]

    intercept = params[0]
    beta = params[1:]

    # Linear predictor
    eta = intercept + X @ beta

    # Keep exponentiation numerically safe
    if not np.isfinite(eta).all():
        raise FloatingPointError("Non-finite linear predictor.")

    # Ratio y / mu, computed on the log scale
    ratio = np.exp(np.log(y) - eta)

    if not np.isfinite(ratio).all():
        raise FloatingPointError("Non-finite cost ratio.")

    # Half the mean Gamma deviance
    data_loss = np.mean(
        ratio - 1 - np.log(ratio)
    )

    # L2 regularization
    penalty = 0.5 * lam * np.dot(beta, beta)

    loss = float(data_loss + penalty)

    # Gradient with respect to the linear predictor
    residual = 1.0 - ratio

    # Intercept gradient
    grad_intercept = np.mean(residual)

    # Coefficient gradient
    grad_beta = (
        np.asarray(X.T @ residual).ravel() / n
        + lam * beta
    )

    gradient = np.concatenate([
        [grad_intercept],
        grad_beta
    ])

    return loss, gradient


In [12]:

# Section 3.3: Verify Gamma Gradient Numerically

from scipy.optimize import check_grad

X_small = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0]
])

y_small = np.array([1.0, 2.0, 3.0])

params_small = np.array([
    0.2,   # Intercept
    0.1,   # First coefficient
    -0.1   # Second coefficient
])

lam = 0.1

def objective_only(params):
    return gamma_loss_and_gradient(
        params, X_small, y_small, lam
    )[0]

def gradient_only(params):
    return gamma_loss_and_gradient(
        params, X_small, y_small, lam
    )[1]

error = check_grad(
    objective_only,
    gradient_only,
    params_small
)

print("Gradient-check error:", error)


Gradient-check error: 8.591180626857736e-09


In [13]:

# Section 3.4: Verify Gamma Objective on Sparse Data

import numpy as np

model_name = "meta/llama-2-70b-chat"

# Use the first 100 sub-training observations
X_small_sparse = X_subtrain[:100]
y_small = cost_subtrain[model_name].to_numpy()[:100]

# Initialize intercept to log of mean observed cost
initial_intercept = np.log(y_small.mean())

# Initialize all feature coefficients to zero
initial_params = np.zeros(X_small_sparse.shape[1] + 1)
initial_params[0] = initial_intercept

# Example regularization strength
lam = 0.1

loss, gradient = gamma_loss_and_gradient(
    initial_params,
    X_small_sparse,
    y_small,
    lam
)

print("Sparse matrix shape:", X_small_sparse.shape)
print("Parameter vector shape:", initial_params.shape)
print("Gradient shape:", gradient.shape)
print("Initial loss:", loss)
print("Gradient norm:", np.linalg.norm(gradient))

assert gradient.shape == initial_params.shape
assert np.isfinite(loss)
assert np.isfinite(gradient).all()

print("\nSparse Gamma objective and gradient verified.")


Sparse matrix shape: (100, 50001)
Parameter vector shape: (50002,)
Gradient shape: (50002,)
Initial loss: 0.0727165232862394
Gradient norm: 0.3958417703385194

Sparse Gamma objective and gradient verified.


In [14]:

# Section 4.1: Test L-BFGS Optimization on Sparse Gamma Regression

import numpy as np
from scipy.optimize import minimize

model_name = "meta/llama-2-70b-chat"

# Small subset for initial optimizer test
X_fit = X_subtrain[:1000]
y_fit = cost_subtrain[model_name].to_numpy()[:1000]

# Initialize intercept at log mean cost
initial_params = np.zeros(X_fit.shape[1] + 1)
initial_params[0] = np.log(y_fit.mean())

# Example regularization strength
lam = 0.1

initial_loss, _ = gamma_loss_and_gradient(
    initial_params, X_fit, y_fit, lam
)

result = minimize(
    fun=gamma_loss_and_gradient,
    x0=initial_params,
    args=(X_fit, y_fit, lam),
    method="L-BFGS-B",
    jac=True,
    options={
        "maxiter": 50,
        "ftol": 1e-9
    }
)

print("Initial objective:", initial_loss)
print("Final objective:", result.fun)
print("Iterations:", result.nit)
print("Converged:", result.success)
print("Message:", result.message)

assert np.isfinite(result.fun)
assert result.fun <= initial_loss + 1e-10


Initial objective: 0.285179166429334
Final objective: 0.08178546338295706
Iterations: 14
Converged: True
Message: CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL


In [15]:

# Section 4.2: Track Gamma Training and Sub-Validation Loss

import numpy as np
import pandas as pd
from scipy.optimize import minimize

model_name = "meta/llama-2-70b-chat"

X_fit = X_subtrain
X_holdout = X_subval

y_fit = cost_subtrain[model_name].to_numpy()
y_holdout = cost_subval[model_name].to_numpy()

lam = 0.1

# Initialize coefficients
initial_params = np.zeros(X_fit.shape[1] + 1)
initial_params[0] = np.log(y_fit.mean())

training_history = []

def record_losses(params):

    intercept = params[0]
    beta = params[1:]

    # Predicted log-costs
    eta_fit = intercept + X_fit @ beta
    eta_holdout = intercept + X_holdout @ beta

    # Positive predicted costs
    mu_fit = np.exp(eta_fit)
    mu_holdout = np.exp(eta_holdout)

    # Unpenalized Gamma deviance
    train_loss = gamma_deviance(y_fit, mu_fit)
    subval_loss = gamma_deviance(y_holdout, mu_holdout)

    training_history.append({
        "iteration": len(training_history),
        "subtrain_deviance": train_loss,
        "subval_deviance": subval_loss
    })

# Record initial losses
record_losses(initial_params)

# Record losses after each L-BFGS iteration
def callback(params):
    record_losses(params)

result = minimize(
    fun=gamma_loss_and_gradient,
    x0=initial_params,
    args=(X_fit, y_fit, lam),
    method="L-BFGS-B",
    jac=True,
    callback=callback,
    options={
        "maxiter": 100,
        "ftol": 1e-9
    }
)

training_history_df = pd.DataFrame(training_history)

print("Optimization iterations:", result.nit)
print("Converged:", result.success)
print("Message:", result.message)

display(training_history_df)


Optimization iterations: 14
Converged: True
Message: CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH


,iteration,subtrain_deviance,subval_deviance
0,0,0.438509,0.434372
1,1,0.337025,0.363009
2,2,0.146900,0.155695
3,3,0.137594,0.149782
4,4,0.129508,0.140934
5,5,0.123604,0.134583
6,6,0.112909,0.122952
7,7,0.110693,0.120614
8,8,0.110430,0.120396
9,9,0.110423,0.120398


In [16]:

# Section 4.2: Gamma Regression with Early Stopping

import numpy as np
import pandas as pd
from scipy.optimize import minimize

class EarlyStoppingException(Exception):
    """Signal that early stopping has been triggered."""
    pass


def train_gamma_early_stopping(
    X_fit,
    y_fit,
    X_holdout,
    y_holdout,
    lam=0.1,
    max_iter=200,
    patience=5,
    min_delta=1e-6
):
    """
    Fit an L2-regularized Gamma regression with a log link.

    Monitor unpenalized Gamma deviance on sub-training
    and sub-validation data.

    Return the best checkpoint and its loss history.
    """

    y_fit = np.asarray(y_fit, dtype=np.float64)
    y_holdout = np.asarray(y_holdout, dtype=np.float64)

    if lam < 0 or patience < 1 or min_delta < 0:
        raise ValueError("Invalid training hyperparameters.")

    if np.any(y_fit <= 0) or np.any(y_holdout <= 0):
        raise ValueError("Gamma costs must be positive.")

    # Initialize parameters
    initial_params = np.zeros(X_fit.shape[1] + 1)
    initial_params[0] = np.log(y_fit.mean())

    history = []
    best_params = initial_params.copy()
    best_loss = np.inf
    best_iteration = 0
    no_improvement = 0

    # Stable deviance calculation using log-costs
    def compute_deviance(X, y, params):
        eta = params[0] + X @ params[1:]
        log_ratio = np.log(y) - eta
        ratio = np.exp(log_ratio)

        return float(
            2.0 * np.mean(ratio - 1.0 - log_ratio)
        )

    def record_iteration(params):
        nonlocal best_params, best_loss
        nonlocal best_iteration, no_improvement

        iteration = len(history)

        train_loss = compute_deviance(
            X_fit, y_fit, params
        )

        val_loss = compute_deviance(
            X_holdout, y_holdout, params
        )

        if not np.isfinite(train_loss) or not np.isfinite(val_loss):
            raise FloatingPointError("Non-finite training loss.")

        history.append({
            "iteration": iteration,
            "subtrain_deviance": train_loss,
            "subval_deviance": val_loss
        })

        # Always preserve a new minimum;
        # min_delta controls patience, not checkpoint selection.
        if val_loss < best_loss:
            best_loss = val_loss
            best_params = params.copy()
            best_iteration = iteration

        # Count meaningful improvements for patience
        if iteration == 0:
            record_iteration.patience_reference = val_loss
        elif val_loss < record_iteration.patience_reference - min_delta:
            record_iteration.patience_reference = val_loss
            no_improvement = 0
        else:
            no_improvement += 1

        if no_improvement >= patience:
            raise EarlyStoppingException()

    # Record initial losses
    record_iteration(initial_params)

    stopped_early = False
    optimizer_result = None

    try:
        optimizer_result = minimize(
            fun=gamma_loss_and_gradient,
            x0=initial_params,
            args=(X_fit, y_fit, lam),
            method="L-BFGS-B",
            jac=True,
            callback=record_iteration,
            options={
                "maxiter": max_iter,
                "ftol": 1e-12,
                "gtol": 1e-8
            }
        )

    except EarlyStoppingException:
        stopped_early = True

    history_df = pd.DataFrame(history)

    return {
        "params": best_params,
        "best_iteration": best_iteration,
        "best_subval_deviance": best_loss,
        "history": history_df,
        "stopped_early": stopped_early,
        "optimizer_result": optimizer_result
    }


In [17]:

# Section 4.2: Test Gamma Early Stopping — Llama-2-70B

model_name = "meta/llama-2-70b-chat"

gamma_test_run = train_gamma_early_stopping(
    X_fit=X_subtrain,
    y_fit=cost_subtrain[model_name].to_numpy(),
    X_holdout=X_subval,
    y_holdout=cost_subval[model_name].to_numpy(),
    lam=0.1,
    max_iter=200,
    patience=5,
    min_delta=1e-6
)

print("Best iteration:",
      gamma_test_run["best_iteration"])

print("Best sub-validation deviance:",
      gamma_test_run["best_subval_deviance"])

print("Stopped early:",
      gamma_test_run["stopped_early"])

print("Iterations recorded:",
      len(gamma_test_run["history"]) - 1)

print("\nFinal recorded losses:")
display(gamma_test_run["history"].tail())


Best iteration: 8
Best sub-validation deviance: 0.1203960935883405
Stopped early: True
Iterations recorded: 13

Final recorded losses:


,iteration,subtrain_deviance,subval_deviance
9,9,0.110423,0.120398
10,10,0.110437,0.120423
11,11,0.110467,0.120449
12,12,0.110481,0.120463
13,13,0.110482,0.120467


In [18]:

# Section 5.1: Train Gamma Regression Models for All 11 LLMs

import numpy as np
import pandas as pd

GAMMA_LAMBDA = 0.1

gamma_final_models = {}
gamma_training_histories = {}
gamma_training_records = []

for model_name in MODELS:

    print(f"Training: {model_name}")

    result = train_gamma_early_stopping(
        X_fit=X_subtrain,
        y_fit=cost_subtrain[model_name].to_numpy(),
        X_holdout=X_subval,
        y_holdout=cost_subval[model_name].to_numpy(),
        lam=GAMMA_LAMBDA,
        max_iter=200,
        patience=5,
        min_delta=1e-6
    )

    # Save the best checkpoint coefficients
    gamma_final_models[model_name] = result["params"].copy()

    # Save the training history
    gamma_training_histories[model_name] = result["history"].copy()

    gamma_training_records.append({
        "model": model_name,
        "lambda": GAMMA_LAMBDA,
        "best_iteration": result["best_iteration"],
        "best_subval_deviance": result["best_subval_deviance"],
        "iterations_recorded": len(result["history"]) - 1,
        "stopped_early": result["stopped_early"]
    })

gamma_training_summary = pd.DataFrame(gamma_training_records)

print("\nAll 11 Gamma regressions trained.")
display(gamma_training_summary)


Training: WizardLM/WizardLM-13B-V1.2
Training: claude-instant-v1
Training: claude-v1
Training: claude-v2
Training: gpt-3.5-turbo-1106
Training: gpt-4-1106-preview
Training: meta/code-llama-instruct-34b-chat
Training: meta/llama-2-70b-chat
Training: mistralai/mistral-7b-chat
Training: mistralai/mixtral-8x7b-chat
Training: zero-one-ai/Yi-34B-Chat

All 11 Gamma regressions trained.


,model,lambda,best_iteration,best_subval_deviance,iterations_recorded,stopped_early
0,WizardLM/WizardLM-13B-V1.2,0.1,7,0.122929,12,True
1,claude-instant-v1,0.1,8,0.231293,13,True
2,claude-v1,0.1,8,0.224531,13,True
3,claude-v2,0.1,8,0.273232,13,True
4,gpt-3.5-turbo-1106,0.1,7,0.136684,12,True
5,gpt-4-1106-preview,0.1,7,0.372607,12,True
6,meta/code-llama-instruct-34b-chat,0.1,7,0.086083,12,True
7,meta/llama-2-70b-chat,0.1,8,0.120396,13,True
8,mistralai/mistral-7b-chat,0.1,8,0.103922,13,True
9,mistralai/mixtral-8x7b-chat,0.1,8,0.107342,13,True


In [19]:

# Section 6.1: Verify All Trained Gamma Models

import numpy as np
import pandas as pd

verification_records = []

for model_name in MODELS:

    params = gamma_final_models[model_name]
    history = gamma_training_histories[model_name]

    # Verify coefficient dimensions and finiteness
    assert params.shape == (X_subtrain.shape[1] + 1,)
    assert np.isfinite(params).all()

    # Verify saved checkpoint matches best history value
    best_recorded_loss = history["subval_deviance"].min()

    # Generate predictions using saved best checkpoint
    eta = params[0] + X_subval @ params[1:]
    predicted_cost = np.exp(eta)

    assert np.isfinite(predicted_cost).all()
    assert (predicted_cost > 0).all()

    # Recalculate sub-validation Gamma deviance
    actual_cost = cost_subval[model_name].to_numpy()

    verified_loss = gamma_deviance(
        actual_cost,
        predicted_cost
    )

    np.testing.assert_allclose(
        verified_loss,
        best_recorded_loss,
        rtol=1e-8,
        atol=1e-10
    )

    verification_records.append({
        "model": model_name,
        "parameter_count": len(params),
        "best_subval_deviance": verified_loss,
        "verified": True
    })

gamma_verification = pd.DataFrame(verification_records)

display(gamma_verification)

print("\nAll 11 Gamma checkpoints verified successfully.")


,model,parameter_count,best_subval_deviance,verified
0,WizardLM/WizardLM-13B-V1.2,50002,0.122929,True
1,claude-instant-v1,50002,0.231293,True
2,claude-v1,50002,0.224531,True
3,claude-v2,50002,0.273232,True
4,gpt-3.5-turbo-1106,50002,0.136684,True
5,gpt-4-1106-preview,50002,0.372607,True
6,meta/code-llama-instruct-34b-chat,50002,0.086083,True
7,meta/llama-2-70b-chat,50002,0.120396,True
8,mistralai/mistral-7b-chat,50002,0.103922,True
9,mistralai/mixtral-8x7b-chat,50002,0.107342,True



All 11 Gamma checkpoints verified successfully.


In [20]:

# Section 7.1: Save Final Early-Stopped Gamma Models

from pathlib import Path
import numpy as np
import pandas as pd
import joblib

PROJECT_ROOT = Path.home() / "Desktop" / "Erdos-Router"

EXPORT_DIR = PROJECT_ROOT / "saved_models" / "gamma_final"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Combine per-model training histories
all_histories = pd.concat(
    [
        history.assign(model=model_name)
        for model_name, history in gamma_training_histories.items()
    ],
    ignore_index=True
)

# Package the fitted models and training metadata
gamma_final_bundle = {
    "model_type": "gamma_log_link_l2",
    "feature_type": "char_tfidf_50k_plus_length",
    "model_names": list(MODELS),
    "models": gamma_final_models,
    "preprocessing": preprocessing,
    "lambda": GAMMA_LAMBDA,
    "training_summary": gamma_training_summary,
    "feature_count": X_subtrain.shape[1],
    "early_stopping": {
        "patience": 5,
        "min_delta": 1e-6,
        "max_iter": 200,
        "monitor": "subvalidation_gamma_deviance"
    },
    "training_split_file": (
        "saved_models/training_splits/"
        "routerbench_grouped_subtrain_subval_seed42.npz"
    ),
    "prediction_target": "expected_inference_cost",
    "prediction_link": "exponential"
}

# Save model bundle
model_path = (
    EXPORT_DIR / "gamma_tfidf_length_early_stopped.joblib"
)

joblib.dump(
    gamma_final_bundle,
    model_path,
    compress=3
)

# Save training histories
history_path = (
    EXPORT_DIR / "gamma_tfidf_length_training_history.csv"
)

all_histories.to_csv(history_path, index=False)

# Save training summary
summary_path = (
    EXPORT_DIR / "gamma_tfidf_length_training_summary.csv"
)

gamma_training_summary.to_csv(summary_path, index=False)

print("Gamma training artifacts saved successfully!")

for path in [model_path, history_path, summary_path]:
    print(f"{path.name} — {path.stat().st_size / 1024**2:.2f} MiB")
    print("Location:", path)


Gamma training artifacts saved successfully!
gamma_tfidf_length_early_stopped.joblib — 4.38 MiB
Location: /Users/garenchiloyan/Desktop/Erdos-Router/saved_models/gamma_final/gamma_tfidf_length_early_stopped.joblib
gamma_tfidf_length_training_history.csv — 0.01 MiB
Location: /Users/garenchiloyan/Desktop/Erdos-Router/saved_models/gamma_final/gamma_tfidf_length_training_history.csv
gamma_tfidf_length_training_summary.csv — 0.00 MiB
Location: /Users/garenchiloyan/Desktop/Erdos-Router/saved_models/gamma_final/gamma_tfidf_length_training_summary.csv


In [24]:

# Section 7.2: Verify Saved Gamma Model Bundle

import joblib
import numpy as np
from pathlib import Path

model_path = (
    PROJECT_ROOT
    / "saved_models"
    / "gamma_final"
    / "gamma_tfidf_length_early_stopped.joblib"
)

# Reload the bundle from disk
loaded_bundle = joblib.load(model_path)

# Verify model count and feature dimensions
assert len(loaded_bundle["models"]) == 11
assert loaded_bundle["feature_count"] == 50001

# Verify every saved parameter vector
for model_name in MODELS:

    original_params = gamma_final_models[model_name]
    loaded_params = loaded_bundle["models"][model_name]

    np.testing.assert_array_equal(
        original_params,
        loaded_params
    )

    # Compare predictions using a few sub-training questions
    X_sample = X_subtrain[:5]

    original_predictions = np.exp(
        original_params[0]
        + X_sample @ original_params[1:]
    )

    loaded_predictions = np.exp(
        loaded_params[0]
        + X_sample @ loaded_params[1:]
    )

    np.testing.assert_allclose(
        original_predictions,
        loaded_predictions,
        rtol=1e-12,
        atol=1e-15
    )

print("Gamma model bundle loaded successfully.")
print("All 11 parameter vectors verified.")
print("Reloaded predictions match original predictions.")


Gamma model bundle loaded successfully.
All 11 parameter vectors verified.
Reloaded predictions match original predictions.
